# Pearls AQI Predictor -- Exploratory Data Analysis

This notebook explores the daily feature table produced by the feature
pipeline (`aqi_predictor.pipelines.feature_pipeline`) and backfill pipeline
(`aqi_predictor.pipelines.backfill_pipeline`), to sanity-check the data and
motivate the feature-engineering and modelling choices in
`aqi_predictor.features.engineering` / `aqi_predictor.models.trainer`.

**Before running this notebook**, make sure you've backfilled some history:

```bash
python scripts/run_backfill.py --lookback-days 180
```

Everything below reads directly from the configured feature store (local
Parquet by default), so it always reflects your actual data -- there is
nothing hard-coded here.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aqi_predictor.config import DEFAULT_CITIES
from aqi_predictor.features.feature_store import get_feature_store

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

store = get_feature_store()
city_keys = [c.key for c in DEFAULT_CITIES]
df = store.read_features(city_keys=city_keys)
df["date"] = pd.to_datetime(df["date"])
print(f"{len(df)} rows across {df['city_key'].nunique()} cities, "
      f"{df['date'].min().date()} .. {df['date'].max().date()}")
df.head()

## 1. Data completeness

How much history do we actually have per city, and how many days are missing (e.g. a failed API call)?

In [ ]:
coverage = (
    df.groupby("city_name")["date"]
      .agg(first="min", last="max", n_days="count")
)
coverage["calendar_days"] = (coverage["last"] - coverage["first"]).dt.days + 1
coverage["missing_days"] = coverage["calendar_days"] - coverage["n_days"]
coverage

In [ ]:
missing_by_col = df[[
    "us_aqi_mean", "pm2_5_mean", "pm10_mean", "temperature_2m_mean",
    "wind_speed_10m_mean", "us_aqi_mean_lag_7d", "target_3d",
]].isna().mean().sort_values(ascending=False)
missing_by_col.plot(kind="barh", title="Fraction of rows missing, key columns")
plt.xlabel("fraction missing")
plt.tight_layout()
plt.show()

Some missingness is expected and *by design*, not a bug:
- `us_aqi_mean_lag_7d` is NaN for each city's first 7 days (no history yet).
- `target_3d` is NaN for each city's last 3 days (the future hasn't happened yet).

`aqi_predictor.features.engineering.training_frame()` drops these rows before
training, which is why the training pipeline needs *some* buffer of extra
history beyond the plain minimum.

## 2. AQI distribution by city

Which cities in this dataset tend to have worse air quality, and how spread out is it day to day?

In [ ]:
order = df.groupby("city_name")["us_aqi_mean"].median().sort_values(ascending=False).index
data = [df.loc[df["city_name"] == c, "us_aqi_mean"].dropna() for c in order]

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(data, labels=order, vert=True, showfliers=False)
ax.set_ylabel("Daily mean US AQI")
ax.set_title("AQI distribution by city")
for label in ax.get_xticklabels():
    label.set_rotation(30)
    label.set_ha("right")
plt.tight_layout()
plt.show()

## 3. AQI trend over time

Does air quality show a clear seasonal or medium-term trend for each city?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for city in df["city_name"].unique():
    city_df = df[df["city_name"] == city].sort_values("date")
    ax.plot(city_df["date"], city_df["us_aqi_mean"], label=city, alpha=0.85)
ax.axhline(100, color="orange", linestyle="--", linewidth=1, label="Moderate/USG boundary (100)")
ax.axhline(150, color="red", linestyle="--", linewidth=1, label="USG/Unhealthy boundary (150)")
ax.set_ylabel("Daily mean US AQI")
ax.set_title("AQI over time by city")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

## 4. Day-of-week and monthly seasonality

The feature pipeline encodes `day_of_week`, `is_weekend` and cyclical month/day-of-year features -- are these patterns actually present in the data?

In [ ]:
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = df.groupby("day_of_week")["us_aqi_mean"].mean().reindex(range(7))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(dow_labels, by_dow.values, color="#2E5EAA")
axes[0].set_title("Mean AQI by day of week")
axes[0].set_ylabel("Mean US AQI")

by_month = df.groupby("month")["us_aqi_mean"].mean().reindex(range(1, 13))
axes[1].bar(range(1, 13), by_month.values, color="#2E5EAA")
axes[1].set_title("Mean AQI by month")
axes[1].set_xlabel("Month")
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 5. Weather relationships

Wind tends to disperse pollutants; humidity and temperature can influence pollutant formation/trapping. Do we see these relationships in the data?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(df["wind_speed_10m_mean"], df["us_aqi_mean"], s=6, alpha=0.3, color="#2E5EAA")
axes[0].set_xlabel("Mean wind speed (10m)")
axes[0].set_ylabel("Mean US AQI")
axes[0].set_title("Wind speed vs AQI")

axes[1].scatter(df["relative_humidity_2m_mean"], df["us_aqi_mean"], s=6, alpha=0.3, color="#2E5EAA")
axes[1].set_xlabel("Mean relative humidity")
axes[1].set_title("Humidity vs AQI")

axes[2].scatter(df["temperature_2m_mean"], df["us_aqi_mean"], s=6, alpha=0.3, color="#2E5EAA")
axes[2].set_xlabel("Mean temperature (C)")
axes[2].set_title("Temperature vs AQI")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [
    "us_aqi_mean", "pm2_5_mean", "pm10_mean", "carbon_monoxide_mean",
    "nitrogen_dioxide_mean", "ozone_mean", "temperature_2m_mean",
    "relative_humidity_2m_mean", "wind_speed_10m_mean", "surface_pressure_mean",
]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, label="Pearson correlation")
ax.set_title("Correlation matrix: pollutants, weather, and AQI")
plt.tight_layout()
plt.show()

## 6. Pollutant composition by city

Which pollutant tends to dominate the AQI reading in each city? (Useful context for the SHAP feature-importance results in the dashboard.)

In [ ]:
pollutant_cols = ["pm2_5_mean", "pm10_mean", "carbon_monoxide_mean",
                   "nitrogen_dioxide_mean", "sulphur_dioxide_mean", "ozone_mean"]
by_city = df.groupby("city_name")[pollutant_cols].mean()
# Normalise each pollutant to 0-1 across cities so they're visually comparable
# despite very different natural units/scales.
normalised = (by_city - by_city.min()) / (by_city.max() - by_city.min())

fig, ax = plt.subplots(figsize=(11, 5))
normalised.plot(kind="bar", ax=ax)
ax.set_ylabel("Relative level (0-1 across cities)")
ax.set_title("Relative pollutant levels by city")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

## 7. How predictable is tomorrow's AQI from today's? (persistence check)

This motivates the `PersistenceBaseline` reference model in `aqi_predictor.models.trainer`: if a 1-day lag already predicts tomorrow's AQI well, any trained model needs to beat that bar to be worth deploying.

In [ ]:
valid = df.dropna(subset=["us_aqi_mean_lag_1d", "us_aqi_mean"])
corr_1d = valid["us_aqi_mean_lag_1d"].corr(valid["us_aqi_mean"])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(valid["us_aqi_mean_lag_1d"], valid["us_aqi_mean"], s=6, alpha=0.3, color="#2E5EAA")
lims = [0, max(valid["us_aqi_mean_lag_1d"].max(), valid["us_aqi_mean"].max())]
ax.plot(lims, lims, color="red", linestyle="--", linewidth=1, label="y = x")
ax.set_xlabel("Yesterday's AQI")
ax.set_ylabel("Today's AQI")
ax.set_title(f"Day-over-day persistence (corr = {corr_1d:.2f})")
ax.legend()
plt.tight_layout()
plt.show()

## Takeaways to look for

Re-run this notebook against your own backfilled data and check:

1. **Coverage** -- are there large gaps in any city's history? If so, re-run
   `scripts/run_backfill.py` for that city before training.
2. **City-level differences** -- cities with a wide AQI spread and clear
   seasonal/day-of-week structure are exactly where the engineered time and
   lag features in `aqi_predictor.features.engineering` should add the most
   value over a naive persistence forecast.
3. **Weather correlations** -- if wind speed shows a clear negative
   relationship with AQI in your data, expect `wind_speed_10m_mean` /
   `wind_speed_10m_max` to show up in the SHAP feature-importance chart in
   the dashboard.
4. **Persistence correlation** -- a high day-over-day correlation means the
   `persistence` baseline will be hard to beat for day+1; the real test of
   the trained models is whether they still beat it (see the "Model
   performance" table in the Streamlit dashboard) and whether the gap holds
   up at day+2 / day+3, where persistence gets much worse.